In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# --------------------------
# LOAD DATASET
# --------------------------
df = pd.read_csv("movie_metadata.csv")

# --------------------------
# CLEANING
# --------------------------
df["genres"] = df["genres"].astype(str)
df["movie_title"] = df["movie_title"].astype(str).str.strip()
df["actor_1_name"] = df["actor_1_name"].astype(str)
df["actor_2_name"] = df["actor_2_name"].astype(str)
df["actor_3_name"] = df["actor_3_name"].astype(str)

# --------------------------
# CREATE TAGS (IMPORTANT)
# --------------------------
df["tags"] = (
    df["genres"] + " " +
    df["actor_1_name"] + " " +
    df["actor_2_name"] + " " +
    df["actor_3_name"]
)

df["tags"] = df["tags"].str.lower()

# --------------------------
# TF-IDF VECTORIZATION
# --------------------------
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["tags"])

# --------------------------
# COSINE SIMILARITY
# --------------------------
similarity = cosine_similarity(tfidf_matrix)

# --------------------------
# MOOD DETECTION
# --------------------------
def detect_mood(text):
    text = text.lower()
    mood_keywords = {
        "happy": ["happy", "joy", "excited", "glad"],
        "sad": ["sad", "depressed", "heartbroken"],
        "angry": ["angry", "furious", "mad"],
        "romantic": ["love", "romantic"],
        "lonely": ["lonely", "alone"]
    }
    for mood, words in mood_keywords.items():
        for w in words:
            if w in text:
                return mood
    return None

# --------------------------
# MOOD → GENRE MAP
# --------------------------
mood_to_genre = {
    "happy": ["Comedy", "Adventure", "Family"],
    "sad": ["Drama", "Biography"],
    "angry": ["Action", "Thriller"],
    "romantic": ["Romance", "Drama"],
    "lonely": ["Family", "Drama"]
}

# --------------------------
# MAIN RECOMMENDER
# --------------------------
def recommend_movies(user_input):

    text = user_input.lower()

    # 1️⃣ MOOD BASED
    mood = detect_mood(text)
    if mood:
        genres = mood_to_genre[mood]
        filtered = df[df["genres"].str.contains("|".join(genres), case=False, na=False)]
        filtered = filtered.sort_values("imdb_score", ascending=False)
        return "mood", mood, filtered.head(5)

    # 2️⃣ ACTOR BASED
    actor_filtered = df[
        (df["actor_1_name"].str.contains(text, case=False, na=False)) |
        (df["actor_2_name"].str.contains(text, case=False, na=False)) |
        (df["actor_3_name"].str.contains(text, case=False, na=False))
    ]

    if not actor_filtered.empty:
        actor_filtered = actor_filtered.sort_values("imdb_score", ascending=False)
        return "actor", user_input.title(), actor_filtered.head(5)

    # 3️⃣ MOVIE BASED (TF-IDF SIMILARITY)
    normalized_input = text.replace(" ", "").replace("-", "")
    titles = df["movie_title"].str.lower().str.replace(" ", "").str.replace("-", "")

    if normalized_input in titles.values:
        idx = titles[titles == normalized_input].index[0]

        sim_scores = list(enumerate(similarity[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        top_movies = sim_scores[1:6]  # skip same movie

        movie_indices = [i[0] for i in top_movies]
        result = df.iloc[movie_indices]

        return "movie", df.iloc[idx]["movie_title"], result

    return "none", None, None
